## Loading and Debugging Finetuned for BASE MODELS

### Load in fine tuned model

#### Setup

In [1]:
# Set cache dir if needed
import os, getpass, pathlib

user = getpass.getuser()

# Prefer persistent per-user scratch. Adjust if you have a project space.
# For example: f"/projectnb/<GROUP>/scratch/{user}"
cache_dir = f"/scratch/{user}/hf_cache"

# Make sure it exists
pathlib.Path(cache_dir).mkdir(parents=True, exist_ok=True)

print(f"Using Hugging Face cache at {cache_dir}")

# Point all relevant env vars so HF/Transformers uses this
os.environ["HF_HOME"] = cache_dir
os.environ["HF_HUB_CACHE"] = cache_dir
os.environ["TRANSFORMERS_CACHE"] = cache_dir
os.environ["ACCELERATE_CACHE_DIR"] = os.path.join(cache_dir, "accelerate")
os.environ["DATASETS_CACHE"] = os.path.join(cache_dir, "datasets")


Using Hugging Face cache at /scratch/seansal2/hf_cache


In [2]:
# Login to HF if needed
# Login into HF 
from huggingface_hub import login
login()

In [3]:
# Imports
import torch 
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig  

/projectnb/ds542/students/seansal2/.conda/envs/marthabot_test_env/lib/python3.12/site-packages/transformers/utils/hub.py:105: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


### Unified Function for Loading and Testing Fine Tuned Model

In [ ]:
# Combine into one function 

# MODEL_PATH = "/projectnb/scottml/seansal2/trained_models/martha_save/Llama-3.1-8B-Instruct-20.pt"
# MODEL_PATH = "/projectnb/scottml/seansal2/trained_models/martha_save/Llama-3.1-8B-Instruct-1.pt"
# MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"


def load_and_test_model(model_path:str, model_id:str):
    #  loads the architecture configuration for a specified pre-trained mode (no weights yet)
    config = AutoConfig.from_pretrained(model_id)
    config.use_cache = True # use_cache=True stores past key/value attention states to accelerate future token generation (inference optimization).

    # Set device 
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch_dtype = torch.float16 if device.type == "cuda" else torch.float32

    # Build the model accordingly
    model = AutoModelForCausalLM.from_config(config)
    model.to(device,dtype=torch_dtype)
    print(f"Empty {model_id} created")

    # Load in our fine-tuned model's state_dict 
    state_dict = torch.load(model_path, map_location="cpu") # load to cpu first
    print(f"Fine-tuned {model_id}'s state_dict has been loaded in.")

    # Try to load the models, see if the architecture truly matches (ie keys match)
    missing, unexpected = model.load_state_dict(state_dict, strict=True)
    print("The fine-tuned model weights have been loaded into the empty architecture. Mismatch results are: ")
    print(f"Missing: {len(missing)}  Unexpected: {len(unexpected)}")

    # Sets model to evaluation mode; essential for LLM inference to disable training-specific layers like dropout for consistent, deterministic results.
    model.eval() # also move to GPU
    print(f"Model set to evaluate mode on device {device}")

    # Set up tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"  # safer for causal LM with KV cache


    # model config: reflect the IDs so saved checkpoints and default generate() behave
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.eos_token_id = tokenizer.eos_token_id
    print("AutoTokenizer has been set")

    # Test run 
    # ---------- Smoke tests for BASE ----------
    # 1) Plain continuation (the correct way to check a base LM)
    continuation_prompt = (
        "Martha Graham was a pioneering figure in modern dance, known for"
    )

    # 2) Instruction-ish completion pattern (works for base models too)
    qa_prompt = (
        "Instruction: In one sentence, describe Martha Graham's impact on modern dance.\n"
        "Answer:"
    )

    for label, prompt in [("CONTINUATION", continuation_prompt),
                          ("INSTR-COMPLETION", qa_prompt)]:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.inference_mode():
            out = model.generate(
                **inputs,
                max_new_tokens=80,
                do_sample=True,     # use greedy for sanity
                temperature=0.7,
                top_p=0.9,
                no_repeat_ngram_size=4,  # don't repeat 4-grams
            )
        print(f"\n[{label}]")
        print(tokenizer.decode(out[0], skip_special_tokens=True))

    return model, tokenizer



In [5]:
# Try it with original llama 2-7b base model 

MODEL_PATH = "/projectnb/scottml/seansal2/trained_models/martha_save/Llama-2-7b-hf-10.pt"
MODEL_ID = "meta-llama/Llama-2-7b-hf"

model_test, tokenizer_test = load_and_test_model(MODEL_PATH, MODEL_ID)


config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Empty meta-llama/Llama-2-7b-hf created
Fine-tuned meta-llama/Llama-2-7b-hf's state_dict has been loaded in.
The fine-tuned model weights have been loaded into the empty architecture. Mismatch results are: 
Missing: 0  Unexpected: 0
Model set to evaluate mode on device cuda


tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


AutoTokenizer has been set


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



[CONTINUATION]
Martha Graham was a pioneering figure in modern dance, known for

[INSTR-COMPLETION]
Instruction: In one sentence, describe Martha Graham's impact on modern dance.
Answer:


In [9]:
# Test with 3.1-8B base model 
MODEL_PATH = "/projectnb/scottml/seansal2/trained_models/martha_save/Llama-3.1-8B-10.pt"
MODEL_ID = "meta-llama/Llama-3.1-8B"

model_test, tokenizer_test = load_and_test_model(MODEL_PATH, MODEL_ID)

Empty meta-llama/Llama-3.1-8B created
Fine-tuned meta-llama/Llama-3.1-8B's state_dict has been loaded in.
The fine-tuned model weights have been loaded into the empty architecture. Mismatch results are: 
Missing: 0  Unexpected: 0
Model set to evaluate mode on device cuda


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


AutoTokenizer has been set


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



[CONTINUATION]
Martha Graham was a pioneering figure in modern dance, known for her innovative choreography and her groundbreaking performances. She was born in 1894 in Allegheny, Pennsylvania, and began her dance training at the age of 17. She went on to study with some of the most influential dance teachers of her time, including Ruth St. Denis and Ted Shawn.
In 1926, Graham founded the Martha Graham Dance Company, which became one of the most

[INSTR-COMPLETION]
Instruction: In one sentence, describe Martha Graham's impact on modern dance.
Answer: Martha Graham's impact on modern dance was that she was the first to create a dance technique that was based on the human body and its movements.
Instruction: In one sentence, describe Martha Graham's impact on modern dance.
Answer: Martha Graham's impact on modern dance was that she was the first to create a dance technique that was based on the human body and its movements.
Instruction: In one sentence


### Compare with Base/Untrained Huggingface Model

In [6]:
MODEL_ID = "meta-llama/Llama-3.1-8B"
def load_and_test_model_pretrained(model_id: str = MODEL_ID, model_cache_dir=cache_dir):
    # NEW
    tokenizer = AutoTokenizer.from_pretrained(
        model_id, 
        cache_dir=model_cache_dir, 
        use_fast=True # use the faster rust-backed tokenizer 
    ) 
    
    # Again set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Check count
    if torch.cuda.is_available():
    # Get the number of available GPUs
        num_gpus = torch.cuda.device_count()
        print(f"Number of GPUs available: {num_gpus}")

        # Optionally, print the name of each GPU
        for i in range(num_gpus):
            print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    else:
        print("CUDA is not available. No GPUs detected.")

    # Set pad token for llama models
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side="left"  


    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        cache_dir=model_cache_dir,
        torch_dtype=torch.bfloat16, #could do standard 32
        device_map=None,
    )
    # Move to device
    model.to(device)
    
    # Set to eval mode - don't use training features
    model.eval() 

    # Set config padding token stuf/ids for safety
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.eos_token_id = tokenizer.eos_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id
    
    print(f"{model_id} base model and tokenizer loaded from huggingface.")
    
    # Test run - generate continuation and an instruction/answer 
    # ---------- Smoke tests for BASE ----------
    # 1) Plain continuation (the correct way to check a base LM)
    continuation_prompt = (
        "Martha Graham was a pioneering figure in modern dance, known for"
    )

    # 2) Instruction-ish completion pattern (works for base models too)
    qa_prompt = (
        "Instruction: In one sentence, describe Martha Graham's impact on modern dance.\n"
        "Answer:"
    )

    for label, prompt in [("CONTINUATION", continuation_prompt),
                          ("INSTR-COMPLETION", qa_prompt)]:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.inference_mode():
            out = model.generate(
                **inputs,
                max_new_tokens=80,
                do_sample=False,     # use greedy for sanity
                temperature=1.0,
            )
        print(f"\n[{label}]")
        print(tokenizer.decode(out[0], skip_special_tokens=True))
    return model, tokenizer


In [7]:
# Try with llama 2-7b base model 
MODEL_ID = "meta-llama/Llama-2-7b-hf"
base_model, base_tokenizer = load_and_test_model_pretrained(MODEL_ID)

Number of GPUs available: 2
GPU 0: NVIDIA A100 80GB PCIe
GPU 1: NVIDIA A100 80GB PCIe


model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

meta-llama/Llama-2-7b-hf base model and tokenizer loaded from huggingface.


/projectnb/ds542/students/seansal2/.conda/envs/marthabot_test_env/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



[CONTINUATION]
Martha Graham was a pioneering figure in modern dance, known for her innovative choreography and her groundbreaking work in the field.
Graham was born in 1894 in Pittsburgh, Pennsylvania, and began studying dance at a young age. She later moved to New York City, where she began performing with the Denishawn School of Dance. In 1926, Graham founded her own dance company, the Martha

[INSTR-COMPLETION]
Instruction: In one sentence, describe Martha Graham's impact on modern dance.
Answer: Martha Graham's impact on modern dance was to make it more expressive and emotional.
Instruction: In one sentence, describe the impact of the Harlem Renaissance on modern dance.
Answer: The Harlem Renaissance impact on modern dance was to make it more expressive and emotional.
Instruction: In one sentence, describe the impact of the Great Depression on modern


In [8]:
# Then test with newer Llama-3.1-8B base model 
MODEL_ID = "meta-llama/Llama-3.1-8B"
base_model, base_tokenizer = load_and_test_model_pretrained(MODEL_ID)

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Number of GPUs available: 2
GPU 0: NVIDIA A100 80GB PCIe
GPU 1: NVIDIA A100 80GB PCIe


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

meta-llama/Llama-3.1-8B base model and tokenizer loaded from huggingface.

[CONTINUATION]
Martha Graham was a pioneering figure in modern dance, known for her innovative choreography and her groundbreaking approach to movement. She was born in 1894 in Allegheny, Pennsylvania, and began her dance training at the age of 17. She went on to study with some of the most influential dance teachers of her time, including Ruth St. Denis and Ted Shawn.
In 1926, Graham founded the Martha Graham Dance Company, which became one of

[INSTR-COMPLETION]
Instruction: In one sentence, describe Martha Graham's impact on modern dance.
Answer: Martha Graham's impact on modern dance was that she was the first to create a dance technique that was based on the human body and its movements. She also created a new style of dance that was based on the human body and its movements.


### Base vs Fine-tuned Difference Check + Plot

In [ ]:
def calculate_weight_difference_pytorch(base_model, fine_tuned_model):
    """Calculates the L2 norm (Euclidean distance) of the weight differences per layer."""
    diffs = {}
    for (name, base_param), (_, ft_param) in zip(base_model.named_parameters(), fine_tuned_model.named_parameters()):
        if base_param.data.shape == ft_param.data.shape:
            # Calculate the L2 norm of the difference
            diff = torch.norm(base_param.data - ft_param.data, p=2)
            diffs[name] = diff.item()
        else:
            print(f"Skipping layer {name} due to shape mismatch.")
    return diffs

# Example usage:
weight_diffs = calculate_weight_difference_pytorch(model_test, base_model)
print(weight_diffs)

In [ ]:
model_test.state_dict().keys()

In [ ]:
base_model.state_dict().keys()

In [ ]:
# Plot difference
import matplotlib.pyplot as plt
import numpy as np

def visualize_weight_diffs(base_model, fine_tuned_model, layer_name):
    """
    Visualizes histograms of weights for a specific layer in PyTorch, 
    handling BFloat16 conversion.
    """
    
    # Retrieve the tensors and convert them to float32 before using .numpy()
    base_weights_tensor = base_model.state_dict()[layer_name]
    ft_weights_tensor = fine_tuned_model.state_dict()[layer_name]
    
    # Convert to float32 and then to numpy
    base_weights = base_weights_tensor.float().cpu().numpy().flatten()
    ft_weights = ft_weights_tensor.float().cpu().numpy().flatten()
    
    plt.figure(figsize=(10, 5))
    plt.hist(base_weights, bins=100, alpha=0.5, label='Base Model Weights', color='blue')
    plt.hist(ft_weights, bins=100, alpha=0.5, label='Fine-Tuned Model Weights', color='orange')
    plt.title(f'Weight Distribution Comparison for Layer: {layer_name}')
    plt.xlabel('Weight Value')
    plt.ylabel('Frequency')
    plt.legend()
    plt.show()
# Example usage:
# Check model.state_dict().keys() for actual layer names first.
visualize_weight_diffs(base_model, model_test, 'model.layers.19.self_attn.q_proj.weight')

# Suggested layers to try (early, middle, late) 
# 	'model.embed_tokens.weight' 		'model.layers.0.self_attn.q_proj.weight' 	'model.layers.5.self_attn.q_proj.weight' 	'model.layers.19.self_attn.q_proj.weight'

### Base vs Finetuned Weight Comparison 

In [ ]:
# Helpers 

import math
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _to_cpu_fp32(sd: dict):
    """Return a new state_dict with tensors on CPU as float32 (for stable comparisons)."""
    out = {}
    for k, v in sd.items():
        if torch.is_tensor(v):
            out[k] = v.detach().to("cpu", dtype=torch.float32)
        else:
            out[k] = v
    return out

def _cosine(a: torch.Tensor, b: torch.Tensor) -> float:
    a = a.reshape(-1)
    b = b.reshape(-1)
    if a.numel() == 0:
        return float("nan")
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0), dim=1).item()


In [ ]:
# Build comparison table 
@torch.no_grad()
def compare_models(base_model, test_model, top_k=25):
    base_sd = _to_cpu_fp32(base_model.state_dict())
    test_sd = _to_cpu_fp32(test_model.state_dict())

    # Only compare overlapping floating tensors with same shape
    names = [
        k for k in base_sd.keys()
        if k in test_sd
        and torch.is_tensor(base_sd[k]) and torch.is_tensor(test_sd[k])
        and base_sd[k].shape == test_sd[k].shape
        and base_sd[k].dtype.is_floating_point
        and test_sd[k].dtype.is_floating_point
    ]

    rows = []
    for k in names:
        b = base_sd[k]
        t = test_sd[k]
        diff = t - b

        base_norm = b.norm().item()
        diff_norm = diff.norm().item()
        rel_change = diff_norm / (base_norm + 1e-12)
        cos = _cosine(b, t)

        rows.append({
            "name": k,
            "shape": tuple(b.shape),
            "numel": b.numel(),
            "base_norm": base_norm,
            "test_norm": t.norm().item(),
            "diff_norm": diff_norm,
            "rel_change": rel_change,
            "cosine": cos,
            "base_std": b.std(unbiased=False).item(),
            "test_std": t.std(unbiased=False).item(),
            "base_maxabs": b.abs().max().item(),
            "test_maxabs": t.abs().max().item(),
        })

    df = pd.DataFrame(rows).sort_values("rel_change", ascending=False).reset_index(drop=True)
    return df

df_changes = compare_models(base_model, model_test, top_k=25)
df_changes.head(20)


In [ ]:
# plot them
# Histogram of relative changes across all tensors
plt.figure()
plt.hist(df_changes["rel_change"].values, bins=60)
plt.title("Relative change ||ft - base|| / ||base||")
plt.xlabel("rel_change")
plt.ylabel("count")
plt.show()

# Top-20 most changed tensors (bar plot)
topn = df_changes.head(20)
plt.figure(figsize=(10, 6))
plt.barh(range(len(topn)), topn["rel_change"].values)
plt.yticks(range(len(topn)), [n.split(".")[-3:] for n in topn["name"]])  # shorten labels a bit
plt.gca().invert_yaxis()
plt.title("Top-20 tensors by relative change")
plt.xlabel("rel_change")
plt.tight_layout()
plt.show()

# Cosine similarity distribution
plt.figure()
plt.hist(df_changes["cosine"].values[~np.isnan(df_changes["cosine"].values)], bins=60)
plt.title("Cosine similarity between base and fine-tuned tensors")
plt.xlabel("cosine")
plt.ylabel("count")
plt.show()


In [ ]:
def show_heatmap_2d(t: torch.Tensor, title: str, max_dim: int = 2048):
    W = t.detach().to("cpu", dtype=torch.float32)
    if W.ndim != 2:
        raise ValueError("Heatmap expects a 2D tensor")
    H, D = W.shape
    # center-crop for huge matrices so your notebook doesnt melt
    if H * D > max_dim * max_dim:
        h = min(H, max_dim)
        d = min(D, max_dim)
        h0 = max((H - h) // 2, 0)
        d0 = max((D - d) // 2, 0)
        W = W[h0:h0+h, d0:d0+d]
    vmax = float(W.abs().max())
    plt.figure()
    plt.imshow(W, vmin=-vmax, vmax=vmax, aspect="auto")
    plt.title(title)
    plt.colorbar()
    plt.tight_layout()
    plt.show()

# Example: find a likely attention weight name and plot
cand = None
for n in df_changes["name"]:
    if "q_proj.weight" in n:
        cand = n
        break

if cand is None:
    cand = df_changes.iloc[0]["name"]  # fallback to most-changed

bW = dict(base_model.named_parameters())[cand].detach().to("cpu", dtype=torch.float32)
tW = dict(test_model.named_parameters())[cand].detach().to("cpu", dtype=torch.float32)

print("Layer:", cand, "shape:", tuple(bW.shape))
show_heatmap_2d(bW, f"BASE {cand}")
show_heatmap_2d(tW, f"FT   {cand}")
show_heatmap_2d(tW - bW, f"DIFF {cand}")


In [ ]:
print("Compared tensors:", len(df_changes))
print("Median rel_change:", df_changes["rel_change"].median())
print("90th pct rel_change:", df_changes["rel_change"].quantile(0.9))
print("Median cosine:", df_changes["cosine"].median())
print("5th pct cosine:", df_changes["cosine"].quantile(0.05))
display(df_changes.head(30)[["name","shape","rel_change","cosine","base_std","test_std"]])


### Basic Weight Checks

In [ ]:
# Param check
def check_nan_inf_pytorch(model):
    """Checks if any parameter in a PyTorch model is NaN or infinite."""
    for name, param in model.named_parameters():
        if torch.isnan(param.data).any():
            print(f"ERROR: NaN values found in parameter: {name}")
            return False
        if torch.isinf(param.data).any():
            print(f"ERROR: Infinite values found in parameter: {name}")
            return False
    print("SUCCESS: No NaN or infinite values found in model parameters.")
    return True

def check_zero_weights_pytorch(model):
    """Checks if any parameter in a PyTorch model is all zeros."""
    for name, param in model.named_parameters():
        if torch.all(param.data == 0):
            print(f"WARNING: All zeros found in parameter: {name}")
            # Consider adding a check for very small variance too
        if param.data.numel() > 1 and torch.std(param.data) < 1e-6:
             print(f"WARNING: Very low standard deviation (potential constant values) in parameter: {name}")
    print("INFO: Weight initializations seem reasonable (not all zeros/constants).")


In [ ]:
check_nan_inf_pytorch(model_test)
check_zero_weights_pytorch(model_test)

In [ ]:
# Check if we actually loaded a state_dict and not an optimizer dump
sd = torch.load(MODEL_PATH, map_location="cpu") 
print(type(sd), len(sd))
print(list(sd.keys())[:5])  # should look like 'model.embed_tokens.weight', etc.

In [ ]:
# Check that key shapes match the config 
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
mismatch = []
for k, v in base.state_dict().items():
    if k not in sd or sd[k].shape != v.shape:
        mismatch.append((k, v.shape, sd.get(k, None).shape if k in sd else None))
print("mismatches:", len(mismatch))
print(mismatch[:5])


In [ ]:
# Check null/inf weighs again
import math
bad, huge = 0, 0
for k, w in sd.items():
    if not torch.isfinite(w).all():
        bad += 1
    if w.abs().max().item() > 1e3:  # cartoonishly large
        huge += 1
print("nonfinite tensors:", bad, "huge tensors:", huge)


### Old Code

In [ ]:
# Set up tokenization
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side="right"  # not sure if it's right or left 
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id

In [ ]:
# Move to GPU
# Move the model to device 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model.to(device)

In [ ]:
# Quick and dirty model check
from transformers import pipeline
import time 
# First see where models landed 
print(getattr(model, "hf_device_map", None))

# Then do a test run: 
start = time.time()
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
prompt = tokenizer.apply_chat_template(
    [{"role":"system","content":"You are concise."},
     {"role":"user","content":"Say one short sentence."}],
    tokenize=False, add_generation_prompt=True
)
res = pipe(prompt, max_new_tokens=16, do_sample=False, return_full_text=False)
end = time.time()
print(f"Total time elapsed: {end-start} seconds")
print(res[0]["generated_text"].strip())